In [5]:
!pip -q install groq

In [6]:
import os
from getpass import getpass
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("Groq_API")

In [7]:
from groq import Groq
import json
import os

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

MODEL = "openai/gpt-oss-20b"

print("Groq connected successfully!")
print("Model:", MODEL)

Groq connected successfully!
Model: openai/gpt-oss-20b


In [8]:
class InvoiceMemory:

    def __init__(self):
        self.items = []
        self.conversation = []

    def add_item(self, name, price, qty):

        item = {
            "name": name,
            "price": float(price),
            "qty": int(qty)
        }

        self.items.append(item)

        return {
            "status": "success",
            "message": f"Added {qty} x {name} at ₹{price:.2f}",
            "item": item
        }

    def get_items(self):
        return self.items

    def remember(self, message):
        self.conversation.append(message)

    def get_memory(self):

        return {
            "items": self.items,
            "conversation": self.conversation
        }

    def clear(self):

        self.items = []
        self.conversation = []


# Create memory
memory = InvoiceMemory()

print("nvoice memory initialized!")

nvoice memory initialized!


In [9]:
def add_item(name, price, qty):

    result = memory.add_item(
        name,
        price,
        qty
    )

    print("\n TOOL 1 → add_item()")

    print(
        f"   Added: {qty} × {name} "
        f"@ ₹{price:.2f}"
    )

    return result

In [10]:
def compute_total(tax_percent):

    items = memory.get_items()

    if not items:

        return {
            "status": "error",
            "message": "No items found in invoice."
        }

    # -----------------------------
    # Calculate subtotal
    # -----------------------------

    subtotal = 0

    for item in items:

        amount = (
            item["price"] *
            item["qty"]
        )

        subtotal += amount

    # -----------------------------
    # DISCOUNT DECISION
    # -----------------------------

    if subtotal >= 1000:

        discount_percent = 10

    elif subtotal >= 500:

        discount_percent = 5

    else:

        discount_percent = 0

    # -----------------------------
    # Calculate discount
    # -----------------------------

    discount_amount = (
        subtotal *
        discount_percent /
        100
    )

    # -----------------------------
    # Taxable amount
    # -----------------------------

    taxable_amount = (
        subtotal -
        discount_amount
    )

    # -----------------------------
    # GST
    # -----------------------------

    tax_amount = (
        taxable_amount *
        float(tax_percent) /
        100
    )

    # -----------------------------
    # Grand total
    # -----------------------------

    grand_total = (
        taxable_amount +
        tax_amount
    )

    result = {

        "status": "success",

        "subtotal": round(
            subtotal,
            2
        ),

        "discount_percent":
            discount_percent,

        "discount_amount":
            round(
                discount_amount,
                2
            ),

        "tax_percent":
            float(tax_percent),

        "tax_amount":
            round(
                tax_amount,
                2
            ),

        "grand_total":
            round(
                grand_total,
                2
            )
    }

    print("\n🔧 TOOL 2 → compute_total()")

    print(
        f"   Subtotal: ₹{result['subtotal']:.2f}"
    )

    print(
        f"   Discount: {result['discount_percent']}%"
    )

    print(
        f"   Discount Amount: "
        f"₹{result['discount_amount']:.2f}"
    )

    print(
        f"   GST: {result['tax_percent']}%"
    )

    print(
        f"   GST Amount: "
        f"₹{result['tax_amount']:.2f}"
    )

    print(
        f"   Grand Total: "
        f"₹{result['grand_total']:.2f}"
    )

    return result

In [11]:
def display_invoice(total_result):

    print("\n")
    print("=" * 70)
    print("                     INVOICE")
    print("=" * 70)

    print(
        f"{'ITEM':25}"
        f"{'QTY':>8}"
        f"{'PRICE':>15}"
        f"{'AMOUNT':>15}"
    )

    print("-" * 70)

    for item in memory.get_items():

        amount = (
            item["price"] *
            item["qty"]
        )

        print(
            f"{item['name'][:25]:25}"
            f"{item['qty']:>8}"
            f"₹{item['price']:>13.2f}"
            f"₹{amount:>13.2f}"
        )

    print("-" * 70)

    print(
        f"{'Subtotal':>50}"
        f" ₹{total_result['subtotal']:>12.2f}"
    )

    print(
        f"{'Discount (' + str(total_result['discount_percent']) + '%)':>50}"
        f" -₹{total_result['discount_amount']:>11.2f}"
    )

    print(
        f"{'GST (' + str(total_result['tax_percent']) + '%)':>50}"
        f" ₹{total_result['tax_amount']:>12.2f}"
    )

    print("=" * 70)

    print(
        f"{'GRAND TOTAL':>50}"
        f" ₹{total_result['grand_total']:>12.2f}"
    )

    print("=" * 70)

In [12]:
def ask_groq(prompt):

    response = client.chat.completions.create(

        model=MODEL,

        messages=[

            {
                "role": "system",

                "content": """
You are an AI Invoice Assistant.

You are an AGENT, not a normal chatbot.

Your job is to create and manage invoices.

You have two tools:

TOOL 1:
add_item(name, price, qty)

TOOL 2:
compute_total(tax_percent)

You must work in multiple steps.

Your behavior:

1. Understand the user's request.
2. Check the current invoice memory.
3. Decide what action should happen NEXT.
4. Add missing items using add_item.
5. After all requested items are present, use compute_total.
6. Use the result of compute_total.
7. Remember previous turns.
8. Never invent prices or quantities.
9. If information is missing, ask the user.

DISCOUNT RULES:

Subtotal >= ₹1000:
10% discount

Subtotal >= ₹500:
5% discount

Subtotal < ₹500:
No discount

IMPORTANT:

Only add items that are required by the current user request
and are not already present in the invoice memory.

Return ONLY valid JSON.

For adding an item:

{
    "action": "add_item",
    "name": "Notebook",
    "price": 120,
    "qty": 2
}

For calculating:

{
    "action": "compute_total",
    "tax_percent": 18
}

If information is missing:

{
    "action": "ask",
    "message": "What is the price of the notebook?"
}

If the task is complete:

{
    "action": "final",
    "message": "Invoice completed."
}
"""
            },

            {
                "role": "user",
                "content": prompt
            }

        ],

        temperature=0

    )

    return response.choices[0].message.content

In [13]:
def invoice_agent(user_input, max_steps=10):

    # --------------------------------
    # Remember user message
    # --------------------------------

    memory.remember(user_input)

    print("\n")
    print("=" * 70)
    print("AI INVOICE AGENT")
    print("=" * 70)

    print("\n USER:")
    print(user_input)

    # --------------------------------
    # AGENT LOOP
    # --------------------------------

    for step in range(max_steps):

        print("\n" + "-" * 70)

        print(
            f" STEP {step + 1}: "
            "Agent deciding next action..."
        )

        # --------------------------------
        # Current memory/state
        # --------------------------------

        current_state = {

            "invoice_items":
                memory.get_items(),

            "conversation_memory":
                memory.get_memory()

        }

        # --------------------------------
        # Give state to AI
        # --------------------------------

        prompt = f"""

CURRENT INVOICE MEMORY:

{json.dumps(current_state, indent=2)}

CURRENT USER REQUEST:

{user_input}

Decide the NEXT action.

Rules:

- Add every required item.
- Do not add an item twice.
- Use the existing invoice memory.
- When all requested items are present,
  calculate the total.
- Use the GST/tax percentage from the user.
- If tax percentage is missing, ask for it.
- Do not invent missing information.

Return ONLY JSON.
"""

        # --------------------------------
        # Ask AI
        # --------------------------------

        content = ask_groq(prompt).strip()

        # --------------------------------
        # Remove markdown formatting
        # --------------------------------

        if content.startswith("```"):

            content = content.replace(
                "```json",
                ""
            )

            content = content.replace(
                "```",
                ""
            )

            content = content.strip()

        # --------------------------------
        # Parse JSON
        # --------------------------------

        try:

            action = json.loads(content)

        except Exception:

            print(
                "\n ERROR: AI returned invalid JSON"
            )

            print(content)

            return None

        # --------------------------------
        # Display decision
        # --------------------------------

        print("\n AGENT DECISION:")

        print(
            json.dumps(
                action,
                indent=2
            )
        )

        action_type = action.get(
            "action"
        )

        # =================================
        # ACTION 1 → ADD ITEM
        # =================================

        if action_type == "add_item":

            result = add_item(

                action["name"],

                action["price"],

                action["qty"]

            )

            print("\n OBSERVATION:")

            print(
                result["message"]
            )

            # Go back to agent
            # and decide next action

            continue

        # =================================
        # ACTION 2 → COMPUTE TOTAL
        # =================================

        elif action_type == "compute_total":

            result = compute_total(

                action["tax_percent"]

            )

            print("\n OBSERVATION:")

            print(
                json.dumps(
                    result,
                    indent=2
                )
            )

            # Display invoice

            display_invoice(
                result
            )

            return result

        # =================================
        # ACTION 3 → ASK USER
        # =================================

        elif action_type == "ask":

            print("\n AGENT:")

            print(
                action["message"]
            )

            return action["message"]

        # =================================
        # ACTION 4 → FINAL
        # =================================

        elif action_type == "final":

            print("\n AGENT:")

            print(
                action["message"]
            )

            return action["message"]

        # =================================
        # UNKNOWN ACTION
        # =================================

        else:

            print(
                "\n Unknown action:",
                action_type
            )

            return None

    print(
        "\n Agent reached maximum steps."
    )

    return None

In [14]:
memory.clear()

invoice_agent("""
Create an invoice with:

2 notebooks at ₹120 each
3 pens at ₹50 each
1 school bag at ₹900

Apply 18% GST.
""")



AI INVOICE AGENT

 USER:

Create an invoice with:

2 notebooks at ₹120 each
3 pens at ₹50 each
1 school bag at ₹900

Apply 18% GST.


----------------------------------------------------------------------
 STEP 1: Agent deciding next action...

 AGENT DECISION:
{
  "action": "add_item",
  "name": "Notebook",
  "price": 120,
  "qty": 2
}

 TOOL 1 → add_item()
   Added: 2 × Notebook @ ₹120.00

 OBSERVATION:
Added 2 x Notebook at ₹120.00

----------------------------------------------------------------------
 STEP 2: Agent deciding next action...

 AGENT DECISION:
{
  "action": "add_item",
  "name": "Pen",
  "price": 50,
  "qty": 3
}

 TOOL 1 → add_item()
   Added: 3 × Pen @ ₹50.00

 OBSERVATION:
Added 3 x Pen at ₹50.00

----------------------------------------------------------------------
 STEP 3: Agent deciding next action...

 AGENT DECISION:
{
  "action": "add_item",
  "name": "school bag",
  "price": 900,
  "qty": 1
}

 TOOL 1 → add_item()
   Added: 1 × school bag @ ₹900.00

 OBSE

{'status': 'success',
 'subtotal': 1290.0,
 'discount_percent': 10,
 'discount_amount': 129.0,
 'tax_percent': 18.0,
 'tax_amount': 208.98,
 'grand_total': 1369.98}

In [15]:
memory.clear()

invoice_agent("""
Create an invoice for 5 laptops.
Apply 18% GST.
""")



AI INVOICE AGENT

 USER:

Create an invoice for 5 laptops.
Apply 18% GST.


----------------------------------------------------------------------
 STEP 1: Agent deciding next action...

 AGENT DECISION:
{
  "action": "ask",
  "message": "What is the price of each laptop?"
}

 AGENT:
What is the price of each laptop?


'What is the price of each laptop?'

In [17]:
print("=" * 70)
print("           AI INVOICE ASSISTANT")
print("=" * 70)

print("""
Commands:
  clear → Start a new invoice
  exit  → Stop the assistant
""")

while True:

    user_input = input("\nYou: ")

    # -----------------------------
    # EXIT
    # -----------------------------

    if user_input.lower() == "exit":

        print(
            "\n Agent: Goodbye!"
        )

        break

    # -----------------------------
    # CLEAR MEMORY
    # -----------------------------

    if user_input.lower() == "clear":

        memory.clear()

        print(
            "\n Agent: "
            "Invoice memory cleared."
        )

        continue

    # -----------------------------
    # RUN AGENT
    # -----------------------------

    invoice_agent(
        user_input
    )

           AI INVOICE ASSISTANT

Commands:
  clear → Start a new invoice
  exit  → Stop the assistant


You: pencil - 200


AI INVOICE AGENT

 USER:
pencil - 200

----------------------------------------------------------------------
 STEP 1: Agent deciding next action...

 AGENT DECISION:
{
  "action": "ask",
  "message": "What is the quantity of pencils?"
}

 AGENT:
What is the quantity of pencils?

You: 300


AI INVOICE AGENT

 USER:
300

----------------------------------------------------------------------
 STEP 1: Agent deciding next action...

 AGENT DECISION:
{
  "action": "add_item",
  "name": "pencil",
  "price": 200,
  "qty": 300
}

 TOOL 1 → add_item()
   Added: 300 × pencil @ ₹200.00

 OBSERVATION:
Added 300 x pencil at ₹200.00

----------------------------------------------------------------------
 STEP 2: Agent deciding next action...

 AGENT DECISION:
{
  "action": "ask",
  "message": "What is the price of the laptops?"
}

 AGENT:
What is the price of the laptops?

You: